# Day 37 — Regularization & linear models
Objectives:
- Ridge vs Lasso vs ElasticNet.
- Coefficient shrinkage.
- Tuning regularization strength.


In [ ]:
import numpy as np
from sklearn.linear_model import Ridge, Lasso
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
X,y = load_diabetes(return_X_y=True)
Xtr,Xte,ytr,yte = train_test_split(X,y,random_state=42)
ridge = Pipeline([('sc', StandardScaler()), ('m', Ridge(alpha=1.0))])
lasso = Pipeline([('sc', StandardScaler()), ('m', Lasso(alpha=0.01, max_iter=10000))])
ridge.fit(Xtr,ytr).score(Xte,yte), lasso.fit(Xtr,ytr).score(Xte,yte)


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — regularization strength, coefficient shrinkage, sparsity, and stability

### Mental model

Linear models minimize a data-fitting loss. Regularization adds a cost
for coefficient size: Ridge uses squared coefficients (L2), Lasso uses
absolute coefficients (L1), and Elastic Net combines them. Increasing
`alpha` gives the penalty more influence, usually increasing bias while
reducing variance.

The penalty acts on coefficient magnitudes, so feature units matter.
Standardization makes one unit of coefficient more comparable across
numeric features. Lasso's zero coefficients are a property of the
fitted sample and penalty, not proof that excluded features are
irrelevant or causally unimportant.

### Read the API before running it

- **`Pipeline([('scale', StandardScaler()), ('model', Ridge(...))])`:** learns scaling inside each training boundary before applying the penalty.
- **`alpha`:** controls penalty strength; compare it on a logarithmic scale with cross-validation.
- **`model.coef_`:** contains coefficients in transformed feature space; inspect magnitude and stability alongside validation error.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — trace Ridge shrinkage as alpha grows

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** The same scaled design and response are used for every alpha so only regularization changes.

In [ ]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

X, y = make_regression(n_samples=250, n_features=8, noise=20, random_state=3701)
Xs = StandardScaler().fit_transform(X)
norms = {}
for alpha in (0.01, 1.0, 100.0):
    coef = Ridge(alpha=alpha).fit(Xs, y).coef_
    norms[alpha] = np.linalg.norm(coef)
print(norms)
assert norms[0.01] > norms[1.0] > norms[100.0]

**Expected observation:** The L2 norm of the coefficient vector decreases as penalty strength increases.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — distinguish Lasso sparsity from stable selection

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** The optimization converged; warnings and `n_iter_` were inspected rather than ignored.

In [ ]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

X, y = make_regression(n_samples=120, n_features=20, n_informative=5,
                       noise=25, random_state=3702)
Xs = StandardScaler().fit_transform(X)
for alpha in (0.1, 1.0, 10.0):
    coef = Lasso(alpha=alpha, max_iter=20_000).fit(Xs, y).coef_
    print(alpha, {"nonzero": np.count_nonzero(coef),
                  "largest_abs": np.max(np.abs(coef))})

**Expected observation:** Stronger L1 regularization generally produces more exact zeros, but selected columns can change with data and alpha.

### Debugging and practice ramp

**Common mistake:** Comparing penalized coefficients from unscaled features and interpreting the largest numeric coefficient as most important.

**Diagnostic:** Inspect feature scales, convergence warnings, coefficient paths, fold scores, and selection frequency across resamples.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define regularization strength, coefficient shrinkage, sparsity, and stability in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not claim feature selection stability from one split or choose alpha from final test performance.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Learner exercises and progressive hints

1. Sweep `alpha` values and plot validation scores for Ridge and Lasso.

**Verify:** For task `Sweep alpha values and plot validation scores for Ridge and Lasso`, show the labeled figure and reconcile it with a numeric summary so appearance is not the only check.






2. Inspect fitted coefficients and compare their sparsity.

The separate solution also demonstrates Elastic Net as a useful extension.

**Verify:** For task `Inspect fitted coefficients and compare their sparsity`, use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed; then report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels.







### Progressive hints

1. Use a logarithmic grid and a log-scaled x-axis because meaningful penalty
   strengths often span orders of magnitude. Keep folds identical.
2. Fit the chosen pipelines on the same training data. Access the final model
   through `named_steps` and count values equal or very close to zero.

### Additional mastery practice

Connect regularization strength to coefficient geometry, validation, and feature scaling. Sparse or stable coefficients are model behaviors, not causal truths.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

3. **Prediction:** Predict the coefficient and training-error behavior of Ridge as alpha moves from nearly zero to an extremely large value. Identify what happens to an unpenalized intercept.
   **Progressive hint:** Larger alpha increases shrinkage and bias; most standard estimators exclude the intercept from the penalty.

**Verify:** For task `Prediction: Predict the coefficient and training-error behavior of Ridge as alpha moves from...`, report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels.







4. **Elastic Net implementation:** Build a scaled ElasticNetCV pipeline, state what alpha and l1_ratio control, and inspect both validation behavior and coefficient sparsity.
   **Progressive hint:** Scaling belongs before the estimator; l1_ratio=1 is Lasso-like and 0 is Ridge-like, while alpha controls overall penalty strength.

**Verify:** For task `Elastic Net implementation: Build a scaled ElasticNetCV pipeline, state what alpha and l1rati...`, assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior; then record the exact command/input, terminal result or returned value, and repeat the critical check from a clean process or fresh state.







5. **Scaling bug:** Fit Lasso to one feature measured in dollars and another measured in millions of dollars. Explain why the penalty treats them unfairly without scaling and repair the comparison.
   **Progressive hint:** The L1 penalty operates on coefficient magnitude; rescaling a feature changes the coefficient needed for the same prediction.

**Verify:** For task `Scaling bug: Fit Lasso to one feature measured in dollars and another measured in millions of...`, state one precise claim, the evidence supporting it, the governing assumption, and a counterexample or limitation; then report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels.







6. **Stability investigation:** Create two highly correlated predictors, refit Lasso across several bootstrap samples, and compare selected features with Ridge predictions.
   **Progressive hint:** Lasso may alternate which correlated feature receives weight; Ridge often distributes weight while predictions remain similar.

**Verify:** For task `Stability investigation: Create two highly correlated predictors, refit Lasso across several...`, record the seed, resampling unit, run count, estimate, and an analytic or hand-worked comparison with a stated tolerance; then use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed.






Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 3 — Prediction


# Practice 4 — Elastic Net implementation


# Practice 5 — Scaling bug


# Practice 6 — Stability investigation
